# Three-dimensional geodesic deposition

This compact spherical workflow exactly deposits a tetrahedral sheet source onto a layered geodesic target mesh. Increase the beam, ray, and angular counts in the configuration for production studies.

In [ ]:
from itertools import combinations
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.collections import PolyCollection
from matplotlib.colors import LogNorm
from mpl_toolkits.mplot3d.art3d import Line3DCollection

from pyGATH.fields import (
    build_geodesic_deposition_mesh_from_grid,
    deposit_simplicial_power_to_mesh,
    interpolate_simplicial_fields_batched,
    simplicialise_sheet_fields,
)
from pyGATH.io import load_simulation_config
from pyGATH.plotting import plot_simplicial_mesh

root = Path.cwd().resolve()
if root.name == "examples":
    root = root.parent
simulation = load_simulation_config(
    root / "configs/example_configs/three_dimensional_geodesic_deposition.toml"
)

In [ ]:
with simulation.reporting():
    grid = simulation.build_grid()
    beams = simulation.load_beams()
    initial_rays = simulation.initialize_rays(grid, beams=beams)
    trace = simulation.trace_rays(initial_rays, grid)
    source = simplicialise_sheet_fields(
        trace.sheet_fields, dimension=3, fields="inverse_brems_deposition"
    )
    target = build_geodesic_deposition_mesh_from_grid(grid, maximum_angular_cells=40)
    resolved_deposition = deposit_simplicial_power_to_mesh(
        source,
        target,
        source_batch_size=1024,
        resolve_beam_sheets=True,
    )
    deposition = resolved_deposition.total
print(f"{source.mesh.nsimplices:,} source tetrahedra per sheet")
print(f"{target.ncells:,} target tetrahedra")
print(f"conservation error={deposition.conservation_error:.3e} W")

In [ ]:
TETRAHEDRON_EDGES = np.asarray(tuple(combinations(range(4), 2)), dtype=np.int32)
SLICE_AXES = {
    0: ((1, 2), ("y", "z")),
    1: ((0, 2), ("x", "z")),
    2: ((0, 1), ("x", "y")),
}


def positive_log_norm(arrays, dynamic_range=1.0e6):
    positive = np.concatenate(
        [np.asarray(array)[np.asarray(array) > 0.0] for array in arrays]
    )
    maximum = positive.max()
    minimum = max(positive.min(), maximum / dynamic_range)
    if minimum >= maximum:
        minimum = 0.5 * maximum
    return LogNorm(vmin=minimum, vmax=maximum)


def tetrahedral_plane_sections(vertices, connectivity, fixed_axis, coordinate=0.0):
    vertices = np.asarray(vertices, dtype=np.float64)
    connectivity = np.asarray(connectivity, dtype=np.int32)
    plotted_axes, _ = SLICE_AXES[fixed_axis]
    scale = max(float(np.ptp(vertices, axis=0).max()), np.finfo(float).tiny)
    tolerance = 64.0 * np.finfo(float).eps * scale
    polygons = []
    cell_indices = []

    for cell_index, vertex_indices in enumerate(connectivity):
        tetrahedron = vertices[vertex_indices]
        distance = tetrahedron[:, fixed_axis] - coordinate
        if distance.min() > tolerance or distance.max() < -tolerance:
            continue

        intersections = [
            tetrahedron[index]
            for index in range(4)
            if abs(distance[index]) <= tolerance
        ]
        for first, second in TETRAHEDRON_EDGES:
            if distance[first] * distance[second] < -(tolerance**2):
                fraction = distance[first] / (distance[first] - distance[second])
                intersections.append(
                    tetrahedron[first]
                    + fraction * (tetrahedron[second] - tetrahedron[first])
                )

        unique = []
        for point in intersections:
            if not any(np.linalg.norm(point - other) <= tolerance for other in unique):
                unique.append(point)
        if len(unique) < 3:
            continue

        polygon = np.asarray(unique)[:, plotted_axes]
        centre = polygon.mean(axis=0)
        angle = np.arctan2(polygon[:, 1] - centre[1], polygon[:, 0] - centre[0])
        polygons.append(polygon[np.argsort(angle)] * 1.0e6)
        cell_indices.append(cell_index)

    return polygons, np.asarray(cell_indices, dtype=np.int32)


def plot_cell_slice(axis, sections, cell_values, fixed_axis, norm):
    polygons, cell_indices = sections
    collection = PolyCollection(
        polygons,
        array=np.asarray(cell_values)[cell_indices],
        cmap="viridis",
        norm=norm,
        edgecolors="none",
    )
    axis.add_collection(collection)
    points = np.concatenate(polygons, axis=0)
    axis.set_xlim(points[:, 0].min(), points[:, 0].max())
    axis.set_ylim(points[:, 1].min(), points[:, 1].max())
    axis.set_aspect("equal")
    _, labels = SLICE_AXES[fixed_axis]
    axis.set_xlabel(rf"${labels[0]}$ [$\mu$m]")
    axis.set_ylabel(rf"${labels[1]}$ [$\mu$m]")
    axis.set_title(rf"${'xyz'[fixed_axis]}=0$")
    return collection


target_sections = {
    fixed_axis: tetrahedral_plane_sections(
        target.vertex_positions, target.simplex_connectivity, fixed_axis
    )
    for fixed_axis in range(3)
}
sheet_depositions = [
    resolved_deposition.select(beam_index=0, sheet_index=sheet) for sheet in range(2)
]

## Beam 1 source sheets

In [ ]:
source_stride = max(1, source.mesh.nsimplices // 2500)
figure = plt.figure(figsize=(12, 5))
for sheet in range(2):
    axis = figure.add_subplot(1, 2, sheet + 1, projection="3d")
    plot_simplicial_mesh(
        source,
        beam_index=0,
        sheet_index=sheet,
        simplex_stride=source_stride,
        ax=axis,
    )
    axis.set_title(f"Beam 1, sheet {sheet + 1}")
figure.suptitle(f"Source tetrahedralisations (stride {source_stride})")
figure.tight_layout()

In [ ]:
def sample_source_slice(fixed_axis, resolution=160):
    plotted_axes, _ = SLICE_AXES[fixed_axis]
    positions = np.asarray(source.mesh.vertex_positions[0])
    coordinates = []
    for component in plotted_axes:
        lower = positions[..., component].min()
        upper = positions[..., component].max()
        padding = 0.03 * max(upper - lower, np.finfo(float).tiny)
        coordinates.append(np.linspace(lower - padding, upper + padding, resolution))
    first, second = np.meshgrid(*coordinates, indexing="xy")
    points = np.zeros((first.size, 3), dtype=np.float64)
    points[:, plotted_axes[0]] = first.ravel()
    points[:, plotted_axes[1]] = second.ravel()
    sampled = interpolate_simplicial_fields_batched(
        source, points, point_batch_size=4096
    )
    values = np.asarray(sampled.values)[
        0, :, :, source.selection.inverse_brems_deposition
    ].reshape((source.mesh.nsheets, resolution, resolution))
    inside = np.asarray(sampled.inside)[0].reshape(
        (source.mesh.nsheets, resolution, resolution)
    )
    return (coordinates[0] * 1.0e6, coordinates[1] * 1.0e6), np.ma.masked_where(
        ~inside, values
    )


source_slices = {fixed_axis: sample_source_slice(fixed_axis) for fixed_axis in range(3)}
source_norm = positive_log_norm(
    [
        source_slices[fixed_axis][1][sheet]
        for fixed_axis in range(3)
        for sheet in range(2)
    ]
)
figure, axes = plt.subplots(2, 3, figsize=(15, 9), squeeze=False)
for sheet in range(2):
    for fixed_axis in range(3):
        axis = axes[sheet, fixed_axis]
        coordinates, values = source_slices[fixed_axis]
        image = axis.pcolormesh(
            *coordinates, values[sheet], shading="auto", norm=source_norm
        )
        _, labels = SLICE_AXES[fixed_axis]
        axis.set_xlabel(rf"${labels[0]}$ [$\mu$m]")
        axis.set_ylabel(rf"${labels[1]}$ [$\mu$m]")
        axis.set_aspect("equal")
        axis.set_title(rf"Sheet {sheet + 1}: ${'xyz'[fixed_axis]}=0$")
figure.colorbar(
    image,
    ax=axes.ravel().tolist(),
    label=r"volumetric deposition [W/m$^3$]",
    shrink=0.85,
)
figure.suptitle("Beam 1 volumetric deposition on the source sheets")

## Deposition mesh and mapped power-density slices

In [ ]:
target_stride = max(1, target.ncells // 4000)
sampled_tetrahedra = np.asarray(target.simplex_connectivity)[::target_stride]
sampled_edges = np.unique(
    np.sort(sampled_tetrahedra[:, TETRAHEDRON_EDGES].reshape((-1, 2)), axis=1),
    axis=0,
)
target_positions_um = np.asarray(target.vertex_positions) * 1.0e6
figure = plt.figure(figsize=(8, 7))
axis = figure.add_subplot(111, projection="3d")
axis.add_collection3d(
    Line3DCollection(
        target_positions_um[sampled_edges], colors="0.25", linewidths=0.35, alpha=0.35
    )
)
axis.auto_scale_xyz(
    target_positions_um[:, 0], target_positions_um[:, 1], target_positions_um[:, 2]
)
axis.set_box_aspect((1, 1, 1))
axis.set_xlabel(r"$x$ [$\mu$m]")
axis.set_ylabel(r"$y$ [$\mu$m]")
axis.set_zlabel(r"$z$ [$\mu$m]")
axis.set_title(f"Geodesic deposition mesh (stride {target_stride})")

In [ ]:
sheet_power_density = [np.asarray(result.power_density) for result in sheet_depositions]
sheet_norm = positive_log_norm(sheet_power_density)
figure, axes = plt.subplots(2, 3, figsize=(15, 9), squeeze=False)
for sheet in range(2):
    for fixed_axis in range(3):
        image = plot_cell_slice(
            axes[sheet, fixed_axis],
            target_sections[fixed_axis],
            sheet_power_density[sheet],
            fixed_axis,
            sheet_norm,
        )
        axes[sheet, fixed_axis].set_title(
            rf"Sheet {sheet + 1}: ${'xyz'[fixed_axis]}=0$"
        )
figure.colorbar(
    image,
    ax=axes.ravel().tolist(),
    label=r"volumetric deposition [W/m$^3$]",
    shrink=0.85,
)
figure.suptitle("Beam 1 sheet-resolved deposition on the target mesh")

In [ ]:
total_density = np.asarray(deposition.power_density)
total_norm = positive_log_norm([total_density])
figure, axes = plt.subplots(1, 3, figsize=(15, 4.5))
for fixed_axis, axis in enumerate(axes):
    image = plot_cell_slice(
        axis, target_sections[fixed_axis], total_density, fixed_axis, total_norm
    )
figure.colorbar(image, ax=axes, label=r"volumetric deposition [W/m$^3$]", shrink=0.85)
figure.suptitle("All-beam deposition on the target mesh")

In [ ]:
layer_power = np.bincount(
    target.simplex_radial_layer,
    weights=np.asarray(deposition.cell_power),
    minlength=target.nradial,
)
radii_um = 0.5 * (target.radial_boundaries[:-1] + target.radial_boundaries[1:]) * 1.0e6
plt.figure(figsize=(7, 4.5))
plt.plot(radii_um, layer_power, marker="o")
plt.xlabel(r"radius [$\mu$m]")
plt.ylabel("deposited power per radial layer [W]")
plt.grid(alpha=0.25)